In [1]:
import math
import torch 
import torch.nn as nn
import numpy as np 
import pandas as pd 
import sys
import torch.nn.functional as F
import tqdm
from pathlib import Path
import json
from transformers import AutoConfig, AutoTokenizer, AutoModel
from Bio import SeqIO 
import bioframe as bf

In [2]:
sys.path.append("../GENALM/")
sys.path.append("../GENALM/dnalm") # hack to add dnalm and src to SYSPATH

In [3]:
from model import load_model
from genalg import *
from utils import pad_to_length, cut_seq, calc_ident

# Параметры

In [4]:
RMT_DIR_PATH = 'enformerrmt_weights/'
TOKENIZER_PATH="/mnt/10tb/home/penzar/dnabert_tokenizers/t2t_1000h_multi_32k/"
MODEL_CFG_PATH="/mnt/10tb/home/penzar/dnabert_configs/L24-H1024-A16-V32k-preln-lastln.json"
MODEL_PATH = Path(RMT_DIR_PATH) / 'model_best.pth'
MODEL_EXP = Path(RMT_DIR_PATH) /  'config.json'
ENFORMER_TARGETS_PATH = "https://raw.githubusercontent.com/calico/basenji/0.5/manuscripts/cross2020/targets_human.txt"
REQUIRED_LENGTH = 1280
USE_REVERSE = True

In [5]:
ENFORMER_TARGETS = {'pancreas':
                    ['CAGE:embryonic pancreas cell line:1B2C6',
                    'CAGE:embryonic pancreas cell line:1C3D3',
                    'CAGE:embryonic pancreas cell line:1C3IKEI',
                    'CAGE:embryonic pancreas cell line:2C6',
                    'CAGE:pancreas, adult'],
           'stomach': ['CAGE:stomach, fetal'],
           'intestines': ['CAGE:small intestine, adult, pool1',
                          'CAGE:Intestinal epithelial cells (polarized)',
                          'CAGE:small intestine, fetal'],
           'gall bladder': ['CAGE:gall bladder, adult'],
           'HEK293': ['CAGE:embryonic kidney cell line: HEK293/SLAM untreated',
                      'CAGE:embryonic kidney cell line: HEK293/SLAM infection, 24hr'],
           'ARPE-19': ['CAGE:Retinal Pigment Epithelial Cells']}

In [6]:
TARGETS = ['pancreas']
OFF_TARGETS = ['stomach',
               'intestines',
               'gall bladder',
               'HEK293',
               'ARPE-19']

# Основная часть

In [7]:
model, tokenizer = load_model(exp_config=MODEL_EXP, 
                   tokenizer_path=TOKENIZER_PATH, 
                   model_cfg_path=MODEL_CFG_PATH,
                   weights_path=MODEL_PATH)

input_seq_len: 12120
model_cfg: ./data/configs/L24-H1024-A16-V32k-preln-lastln.json
model_cls: src.gena_lm.modeling_rmt:RMTEncoderForEnformer
backbone_cls: downstream_tasks.enformer.enformer_model:BertForEnformer
input_size: 512
num_mem_tokens: 5
max_n_segments: 24
tokenizer: ./data/tokenizers/t2t_1000h_multi_32k/


In [8]:
device = torch.device("cuda:1")
model.to(device);

In [10]:
df_targets = pd.read_csv(ENFORMER_TARGETS_PATH, sep='\t')
enformer_description = df_targets.description
def map_enformer_pos(t):
    return np.where(enformer_description.str.startswith(t))[0]

In [11]:
enformer_targets_poses = {}
for key, names in ENFORMER_TARGETS.items():
    poses = []
    for t in names:
        
        if (mapping := map_enformer_pos(t)).shape[0] == 0:
            raise Exception(f"Wrong target name {t}")
        poses.extend(mapping)
    enformer_targets_poses[key] = poses

In [14]:
preoptimized_seqs = SeqIO.to_dict(SeqIO.parse("optimized_seqs.fasta", format="fasta"))

In [15]:
preoptimized_seqs

{'seq1': SeqRecord(seq=Seq('CAGTAACGGGAAGCTTTGTTCAGGCTTTGAAGTTCATACCTCTGTGCCTTTGTG...AAG'), id='seq1', name='seq1', description='seq1', dbxrefs=[]),
 'seq2': SeqRecord(seq=Seq('ACCTGGCGCCTACTGTGCCTGTCCCAGCCACCTGTCCCAAACTCACTGGCCTTT...AGA'), id='seq2', name='seq2', description='seq2', dbxrefs=[]),
 'seq3': SeqRecord(seq=Seq('TCTTAGGGAAAATATATTATATATTCGTGTATTAGAGCTGTATATTTGATTTAT...GCC'), id='seq3', name='seq3', description='seq3', dbxrefs=[]),
 'seq4': SeqRecord(seq=Seq('CACTAGCCAGTGGGATTTTAGTCCTACGGCCGCTTTGTGCTTCAGTATCGTAAT...GGC'), id='seq4', name='seq4', description='seq4', dbxrefs=[]),
 'seq5': SeqRecord(seq=Seq('AGCTTAACCTTGGTGTTAGTCACTGCAAGGCCAGGAGATTTTTATCTCGCACAG...AGT'), id='seq5', name='seq5', description='seq5', dbxrefs=[]),
 'seq6': SeqRecord(seq=Seq('TCGCTTATCGGACGGTAAGGCATGGAGAAACAGCGAGTACCAACAGGCCAATAT...AAA'), id='seq6', name='seq6', description='seq6', dbxrefs=[]),
 'seq7': SeqRecord(seq=Seq('TCTTTCCACCGAATCTCCACTTCTGGGTAAAATTGGGTAGGGTCTTCTTTTCTT...AGA'), id='seq7', nam

In [ ]:
blat_res = pd.read_table("blat_preoptimized.txt", skiprows=2, header=None, names=names) # zero-based

In [ ]:
header =[]
rows = []
with open("blat_preoptimized.txt", 'r') as inp:
    for _ in range(2):
        header.append(inp.readline().strip().split('\t'))

In [ ]:
names = header[0][0:1] +[x.strip() + ' ' + y.strip() for x, y in zip(header[0][1:-3], header[1])] + header[0][-3:]
names = [x.strip() for x in names]

In [ ]:
blat_res.head()

In [ ]:
blat_res.columns

In [ ]:
blat_res['tStarts'] = blat_res['tStarts'].str.split(',').apply(lambda x: [int(k) for k in x if k])
blat_res['qStarts'] = blat_res['qStarts'].str.split(',').apply(lambda x: [int(k) for k in x if k])

blat_res['blockSizes'] = blat_res['blockSizes'].str.split(',').apply(lambda x: [int(k) for k in x if k])

In [ ]:
formatted_blat = []
for _, row in blat_res.iterrows():
    for st, sz, qst in zip(row['tStarts'], row['blockSizes'], row['qStarts']):
        new_row = {'chrom': row['T name'], 'start': st, 'end': st+sz, 'strand': row['strand'], 'seq_name': row['Q name'], 'seq_start': qst, 'seq_end': qst+sz}
        formatted_blat.append(new_row)

In [ ]:
formatted_blat = pd.DataFrame(formatted_blat)
formatted_blat

In [ ]:
for _, row in formatted_blat.iterrows():
    target_codon = 'ATG'
        
    codons = set(''.join(x) for x in list(product('ATGC', 'ATGC', 'ATGC'))) - set([target_codon]) 
    seq = removed_region_atg[row.seq_name]
    
    for pos in tqdm.tqdm(range(row.seq_start, row.seq_end)):
        if seq[pos:pos+3] != target_codon:
            continue
    
        variants = []
        for cod in codons:
            new_seq = seq[:pos] + cod + seq[pos+3:]
            if target_codon in new_seq[pos-2:pos+5]:
                continue
            sc = get_score(model=model, seq=new_seq, criterion=criterion, device=device, method='remove_atg')
            variants.append(sc)
        seq = min(variants, key=lambda x: x.score).seq
    removed_region_atg[row.seq_name] = seq